In [ ]:
#svc: 숫자를 분류하는 머신러닝 모델
#metrics: 예측 결과의 정답률 계산
#ImageOps: 이미지 색상 반전
#np 이미지를 숫자 배열로 변환
from sklearn import model_selection,svm,metrics 
import pandas as pd
from PIL import Image,ImageOps
import numpy as np

In [ ]:
#데이터 가져오기
train_csv = pd.read_csv('data/train.csv',header=None)   #공부할 데이터  / header=None: 파일 첫 줄도 데이터로 읽고, 열이름으로 처리x
test_csv = pd.read_csv('data/t10k.csv',header=None)     #모델의 실력을 확인할 데이터

In [ ]:
# 픽셀의 정규화 / 0-255 숫자를 0-1사이의 숫자로 정규화(일정한 범위로 맞추는 것)
# x =이미지 한장의 픽셀값784개
def test(x):
    output = []
    for i in x:
        output.append(float(i)/256)     #실수로 바꾸고 256으로 나눈 뒤 리스트 추가
    return output

In [ ]:
#정답열 제외한 픽셀값 가져오기 iloc[행,열]- 위치번호 선택
train_csv.iloc[:,1:].values       #[:,1:]- 모든 행,두번째 열부터 마지막 열까지 / values:선택한 표의 값을 numpy배열로 꺼냄

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1001, 784))

In [ ]:
# 데이터 정규화 : 0-255
#iloc[0,1:] - 첫번째 행에서 두번째 열부터 마지막 열까지 가져온다.(가로 첫줄에서 맨 앞 칸을 뺀 나머지 값을 가져오는것)
#map(test,이미지들의 픽셀값) = 테스트함수를 이미지마다 적용
train_csv_data =list(map(test,train_csv.iloc[:,1:].values))
test_csv_data = list(map(test,test_csv.iloc[:,1:].values))


In [248]:
# 데이터 전처리 끝
# 정답 label 작업
train_csv_label = train_csv[0].values
train_csv_label
test_csv_label = test_csv[0].values
test_csv_label[0]

np.int64(7)

In [ ]:
#데이터학습
clf =svm.SVC()  #다른알고리즘을 사용해도 됨.
clf.fit(train_csv_data,train_csv_label)

#데이터예측
result = clf.predict(test_csv_data)

#정답률
#score= clf.score(test_csv_data,test_csv_label)
score = metrics.accuracy_score(test_csv_label,result)
print('정답률:',score)    #실제 정답과 예측값을 비교해 맞힌 비율 구함

정답률: 0.8842315369261478


In [ ]:
# 이미지 열기, 흑백으로 변환
img = Image.open('data/a9.png').convert('L')

# 색상반전 : 0-검정,255-흰색  
img = ImageOps.invert(img)

# 숫자가 있는 영역만 추출
# getbbox()는 0이 아닌 픽셀이 있는 영역을 둘러싸는 사각형
# crop: 찾은 사각형대로 이미지 자르기
bbox = img.getbbox()
img = img.crop(bbox)

#비율축소 -사이즈20*20 / LANCZOS-이미지를 줄일 때 픽셀값을 계산하는 방식
img.thumbnail((20,20),Image.Resampling.LANCZOS)
img

#비울확대 -사이즈28*28 /'L':밝기 하나로 픽셀을 표현하는 회색조 모드, 0: 배경밝기
new_img = Image.new('L',(28,28),0)

#배경사이즈 숫자를 가운데 붙여넣기
#paste: 이미지 붙여넣기
x = (28-img.width)//2
y = (28-img.height)//2
new_img.paste(img,(x,y))

# 예측용 데이터 - 0-255/0-1
# 이미지를 실수형Numpy배열로 바꿈
# reshape: 배열의 모양 바꾸기
img_data= np.array(new_img,dtype=np.float32)
flat_img_data = img_data.reshape(1,784)/256.0

#데이터예측 - 2차원 형태여야 예측이 됨.
result = clf.predict(flat_img_data)
print('예측번호 : ',result)

In [252]:
#new_img.resize((280,280)).show()

In [253]:
# input ="0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 18 18 18 126 136 175 26 166 255 247 127 0 0 0 0 0 0 0 0 0 0 0 0 30 36 94 154 170 253 253 253 253 253 225 172 253 242 195 64 0 0 0 0 0 0 0 0 0 0 0 49 238 253 253 253 253 253 253 253 253 251 93 82 82 56 39 0 0 0 0 0 0 0 0 0 0 0 0 18 219 253 253 253 253 253 198 182 247 241 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 80 156 107 253 253 205 11 0 43 154 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 14 1 154 253 90 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 139 253 190 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 11 190 253 70 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 35 241 225 160 108 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 81 240 253 253 119 25 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 45 186 253 253 150 27 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 16 93 252 253 187 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 249 253 249 64 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 46 130 183 253 253 207 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 39 148 229 253 253 253 250 182 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 24 114 221 253 253 253 253 201 78 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 23 66 213 253 253 253 253 198 81 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 18 171 219 253 253 253 253 195 80 9 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 55 172 226 253 253 253 253 244 133 11 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 136 253 253 253 212 135 132 16 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0"
# input = input.split(' ')

# # 28 x 28 손글씨 이미지 흑백처리해서 255color코드로 변경

# for i in range(len(input)):
#     print('{:3}'.format(input[i]),end='')
#     if i%28 == 0:
#         print()